# 01 — Data Collection

Загружаем городскую инфраструктуру из OpenStreetMap и сохраняем в `data/raw/`.

Что должно быть в ноутбуке:
- выбор города и bounding box / place name
- выбор категорий OSM (cafe, restaurant, pharmacy, bar, ...)
- загрузка через `osmnx.features_from_place`
- базовая очистка (координаты, name, category)
- сохранение в CSV / GeoJSON

In [305]:
import osmnx as ox
import pandas as pd
from pathlib import Path

def show_all_columns(q):
    column_report = pd.DataFrame({
        'Column_Name': q.columns,
        'Non_Null_Count': q.count().values,
        'Fill_Rate_%': (q.count().values / len(q)) * 100,
        'Dtype': q.dtypes.values,
        'Example_Value': [q[col].dropna().iloc[0] if not q[col].dropna().empty else "---" for col in q.columns]
    })
    column_report = column_report.sort_values('Non_Null_Count', ascending=False)
    with pd.option_context('display.max_rows', None):
        display(column_report)

RAW = Path('../data/raw')
RAW.mkdir(parents=True, exist_ok=True)

In [306]:
# TODO: choose place and categories
place = 'Armenia'
tags = {'amenity': ['cafe', 'restaurant', 'pharmacy', 'bar']}
q = ox.features_from_place(place, tags=tags)
q.shape

show_all_columns(gdf)



/opt/anaconda3/lib/python3.13/site-packages/osmnx/_overpass.py:271: UserWarning: This area is 16 times your configured Overpass max query area size. It will automatically be divided up into multiple sub-queries accordingly. This may take a long time.
  multi_poly_proj = utils_geo._consolidate_subdivide_geometry(poly_proj)


,Column_Name,Non_Null_Count,Fill_Rate_%,Dtype,Example_Value
0,geometry,2782,100.000000,geometry,POINT (44.5248669 40.1783161)
1,amenity,2782,100.000000,object,restaurant
232,lat,2782,100.000000,float64,40.178316
233,lon,2782,100.000000,float64,44.524867
2,name,1950,70.093458,object,Օլդ Մարանի
3,name:en,1093,39.288282,object,Old Marani
29,check_date,1070,38.461538,object,2024-11-01
19,opening_hours,861,30.948958,object,Mo-Su 10:00-00:00
27,addr:street,857,30.805176,object,Ալեք Մանուկյանի փողոց
24,addr:housenumber,749,26.923077,object,7


In [307]:
centroids = gdf.geometry.centroid

/var/folders/62/v111z_vs2tg87mfz2gzmwr4r0000gn/T/ipykernel_34591/3439640891.py:1: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroids = gdf.geometry.centroid


In [308]:
gdf['lat'] = centroids.y
gdf['lon'] = centroids.x

In [ ]:
no_name_gdf = gdf[gdf['name'].isna()]
# есть 100 заведений с name:en и без name
no_name_en_gdf = no_name_gdf[no_name_gdf['name:en'].isna()]
# есть 23 заведения с name:ru и без name:en
no_name_ru_gdf = no_name_en_gdf[no_name_en_gdf['name:ru'].isna()]
no_name_signed_gdf = no_name_ru_gdf[no_name_ru_gdf['name:signed'].isna()]
no_name_hy_gdf = no_name_signed_gdf[no_name_signed_gdf['name:hy'].isna()]
show_all_columns(no_name_hy_gdf)

,Column_Name,Non_Null_Count,Fill_Rate_%,Dtype,Example_Value
0,geometry,697,100.000000,geometry,POINT (44.524653 40.1307286)
1,amenity,697,100.000000,object,pharmacy
232,lat,697,100.000000,float64,40.130729
233,lon,697,100.000000,float64,44.524653
13,healthcare,291,41.750359,object,pharmacy
29,check_date,203,29.124821,object,2026-03-16
107,building,89,12.769010,object,yes
27,addr:street,86,12.338594,object,Տիտոգրադյան փողոց
16,wheelchair,82,11.764706,object,limited
19,opening_hours,80,11.477762,object,Mo-Fr 08:30-23:00
